# 03 · RAG evaluation: retrieval + grounded composition

End-to-end check of Project 1 against a small fixture index built in notebook memory — same composer and FAISS store code as production.

The store needs FAISS; the cell below installs it on demand so the notebook also runs on a bare Colab runtime.

In [ ]:
# --- bootstrap -------------------------------------------------------------
# Working-directory independent, so this notebook runs identically in Jupyter,
# in VS Code and in Google Colab (kernel CWD is /content, and the checkout may
# sit anywhere below it).  Finds the repository, puts services/model-server on
# sys.path, and generates the git-ignored synthetic corpus on first use.
#
# Needs the *repository*, not just this .ipynb: it imports `app` and reads
# `ml/data/generated`.  In a fresh Colab runtime run this once, above or below
# this cell, then re-run:
#
#     !git clone <your-fork-or-url> /content/catalogue-ai
#
# Already cloned elsewhere? Either set CATALOGUE_AI_ROOT=/path/to/catalogue-ai,
# or rely on the scan below.
import os, sys, pathlib

_HINT = pathlib.Path("ml/notebooks/nbsetup.py")


def _is_repo(path):
    return (path / _HINT).is_file()


def _candidates():
    """Ordered places a checkout might be, nearest first."""
    out = []
    env = os.environ.get("CATALOGUE_AI_ROOT")
    if env:
        out.append(pathlib.Path(env).expanduser())
    cwd = pathlib.Path.cwd().resolve()
    out.extend([cwd, *cwd.parents])
    out.extend([pathlib.Path("/content"), pathlib.Path.home()])
    seen, ordered = set(), []
    for p in out:
        try:
            r = p.resolve()
        except OSError:
            continue
        if r not in seen:
            seen.add(r)
            ordered.append(r)
    return ordered


def _find_repo():
    for c in _candidates():
        if _is_repo(c):
            return c
    # shallow scan: a Colab clone can land in any folder below /content,
    # including below a Drive mount
    frontier = [pathlib.Path("/content"), pathlib.Path.home(),
                pathlib.Path.cwd().resolve()]
    for _ in range(4):
        nxt = []
        for d in frontier:
            if _is_repo(d):
                return d
            try:
                kids = [p for p in sorted(d.iterdir())
                        if p.is_dir() and not p.name.startswith(".")
                        and p.name != "__pycache__"]
            except (OSError, PermissionError):
                kids = []
            nxt.extend(kids)
        frontier = nxt
        if not frontier:
            break
    return None


REPO_ROOT = _find_repo()
if REPO_ROOT is None:
    raise SystemExit(
        "catalogue-ai repository not found.\n\n"
        "This notebook needs the repository (it imports `app` and reads\n"
        "ml/data/generated), not just this .ipynb file. In a fresh Colab\n"
        "runtime run this once, then re-run this cell:\n\n"
        "    !git clone <your-fork-or-url> /content/catalogue-ai\n\n"
        "Already cloned elsewhere? Point the notebook at it:\n\n"
        "    import os; os.environ['CATALOGUE_AI_ROOT'] = '/path/to/catalogue-ai'\n\n"
        "searched:\n  " + "\n  ".join(str(c) for c in _candidates()))

sys.path.insert(0, str(REPO_ROOT / "ml" / "notebooks"))
import nbsetup

_summary = nbsetup.bootstrap()
REPO_ROOT = pathlib.Path(_summary["repo_root"])
DATA_DIR = pathlib.Path(_summary["data_dir"])

In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec('faiss') is None:
    print('installing faiss-cpu ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'faiss-cpu'], check=True)
else:
    print('faiss already available')

In [ ]:
import json
import numpy as np
from app.faiss_store import FaissStore
from app.answer_composer import AnswerComposer
from app.embed import HashEmbedder

emb = HashEmbedder(dim=64)
store = FaissStore(dim=64)
rows = [json.loads(l) for l in (DATA_DIR / 'train.jsonl').open()][:120]
print('corpus rows:', len(rows))

In [ ]:
# index descriptions as chunk evidence
entries = []
for i, r in enumerate(rows):
    vec = emb.encode([r['description']])[0]
    entries.append(dict(faiss_id=i, item_id=r['item_id'], modality='text',
                        text=r['description'], vector=vec.tolist()))
store.upsert(entries)
print('indexed:', store.counts())

In [ ]:
# retrieval: ask about a known feature
q = f"Which item features {rows[0]['spec']['features'][0]}?"
hits = store.search(text_vector=emb.encode([q])[0], top_k=6)
print('question:', q)
print('top-1 item:', hits[0]['item_id'], 'gold:', rows[0]['item_id'],
      'score:', round(hits[0]['score'], 3))

In [ ]:
# grounded composition + citation gate
composer = AnswerComposer(0.35)
out = composer.compose(q, hits, mode='extractive')
print('answer:', out['answer'])
print('passed:', out['citation_check']['passed'])

In [ ]:
# the gate: an ungrounded question must refuse to answer
bad = composer.compose('What is the warranty on the lunar rover?', hits)
print('answer:', bad['answer'])
assert bad['citation_check']['passed'] is False
print('citation gate held')

## hit@k / MRR over the generated qrels

`HashEmbedder` maps each input through a SHA-256-seeded RNG, so its
vectors carry **no lexical or semantic signal** — it is a plumbing
fixture for CPU smoke runs, not a retrieval model. The hit@k/MRR
below therefore measure the index, not retrieval quality, and will
sit near zero by design. The token-overlap baseline is shown beside
it as the floor a real encoder (SBERT/CLIP) has to clear.

In [ ]:
import re

qrels = [json.loads(l) for l in (DATA_DIR / 'qrels.jsonl').open()][:50]
TOKEN_RE = re.compile(r'[a-z0-9]+')

def overlap_score(question, description):
    """Jaccard-ish lexical overlap — the no-model baseline."""
    q = set(TOKEN_RE.findall(question.lower()))
    d = set(TOKEN_RE.findall(description.lower()))
    return len(q & d) / max(1, len(q))

def evaluate(rank_fn):
    hit1 = rr = 0.0
    for row in qrels:
        ranked = rank_fn(row['question'])
        rank = next((i + 1 for i, item in enumerate(ranked)
                     if item['item_id'] == row['item_id']), None)
        hit1 += bool(rank and rank <= 1)
        rr += 1 / rank if rank else 0
    return {'hit@1': round(hit1 / len(qrels), 4),
            'mrr': round(rr / len(qrels), 4)}

by_hash = lambda question: store.search(
    text_vector=emb.encode([question])[0], top_k=len(rows))
by_words = lambda question: sorted(
    rows, key=lambda r: -overlap_score(question, r['description']))

print(json.dumps({'qrels': len(qrels),
                  'hash_embedder (plumbing only)': evaluate(by_hash),
                  'token_overlap_baseline': evaluate(by_words)}, indent=2))